# Μοντελοποίηση Μη Γραμμικών Καμπυλών Ζήτησης Λιανικής με PROC GAM

## Συνοπτική Παρουσίαση

Αυτό το notebook χρησιμοποιεί το **PROC GAM** για να μοντελοποιήσει τις εβδομαδιαίες πωλήσεις μονάδων παντοπωλείου ως μια ομαλή, μη γραμμική συνάρτηση της τιμής ραφιού, της θερμοκρασίας καταστήματος (ένας δείκτης εποχικότητας), και της προωθητικής δαπάνης, με μια παραμετρική επίδραση σημαίας προώθησης. Τα γενικευμένα προσθετικά μοντέλα επιτρέπουν σε έναν διευθυντή κατηγορίας να ανακτήσει τις πραγματικές καμπυλωτές μορφές ελαστικότητας τιμής και εποχιακής ζήτησης που μια γραμμική παλινδρόμηση θα παρέβλεπε, υποστηρίζοντας πιο ακριβείς αποφάσεις τιμολόγησης και προώθησης.

## Πηγές Δεδομένων

| Σύνολο Δεδομένων | Γραμμές | Κλίμακα | Βασικές Μεταβλητές | Περιγραφή |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | εβδομάδα | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Συνθετική εβδομαδιαία σειρά σημείου πώλησης για ένα βασικό κατάστημα παντοπωλείου σε 100 συνεχόμενες εβδομάδες (σχεδόν δύο εποχιακούς κύκλους). Παράγεται ενσωματωμένα με `call streaminit(20250531)` και `rand()`. Οι εβδομαδιαίες πωλήσεις μονάδων ακολουθούν μια διαδικασία ζήτησης Poisson της οποίας ο μέσος όρος καθοδηγείται από μια εκθετική καμπύλη απόκρισης τιμής, μια τετραγωνική επίδραση θερμοκρασίας/εποχικότητας με κορύφωση κοντά στους 72F, και μια κοίλη (τετραγωνική ρίζα) ανάταση δαπάνης προώθησης, συν μια διακριτή σημαία εβδομάδας προώθησης.

# Μοντελοποίηση Μη Γραμμικών Καμπυλών Ζήτησης Λιανικής με PROC GAM

Η ζήτηση λιανικής σπάνια ανταποκρίνεται σε τιμή, καιρό, ή προωθητική δαπάνη με ευθεία γραμμή. Μια μικρή μείωση τιμής σε ένα βασικό είδος μπορεί να μετακινήσει ελάχιστα τον όγκο, ενώ η υπέρβαση ενός ψυχολογικού σημείου τιμής μπορεί να προκαλέσει απότομο άλμα· η ζήτηση που καθοδηγείται από τον καιρό κορυφώνεται σε ένα άνετο ενδιάμεσο εύρος και πέφτει και στα δύο άκρα· και η προωθητική ανάταση δείχνει φθίνουσες αποδόσεις καθώς αυξάνεται η δαπάνη.

**PROC GAM** (γενικευμένα προσθετικά μοντέλα) προσαρμόζει κάθε παράγοντα με τον δικό του ομαλό όρο spline, ώστε τα δεδομένα — όχι μια άκαμπτη γραμμική υπόθεση — να καθορίζουν τη μορφή κάθε καμπύλης ζήτησης. Εδώ μοντελοποιούμε τις εβδομαδιαίες πωλήσεις μονάδων για ένα μοναδικό βασικό κατάστημα παντοπωλείου σε 100 συνεχόμενες εβδομάδες, συνδυάζοντας μια παραμετρική σημαία προώθησης με splines εξομάλυνσης στην τιμή, τη θερμοκρασία, και τη δαπάνη προώθησης υπό μια απόκριση Poisson.

## Βήμα 1 — Παραγωγή μιας συνθετικής εβδομαδιαίας σειράς πωλήσεων

Προσομοιώνουμε 100 συνεχόμενες εβδομάδες (σχεδόν δύο εποχιακούς κύκλους) δεδομένων σημείου πώλησης για ένα βασικό κατάστημα. Η διαδικασία παραγωγής δεδομένων είναι σκόπιμα μη γραμμική ώστε να μπορούμε να επιβεβαιώσουμε ότι το GAM ανακτά ρεαλιστικές μορφές:

- Η **Τιμή** καθοδηγεί τον όγκο μέσω μιας εκθετικής καμπύλης απόκρισης (`exp(-1.1 * Price)`), οπότε η ζήτηση αυξάνεται απότομα καθώς η τιμή πέφτει.
- Η **Θερμοκρασία** λειτουργεί ως δείκτης εποχικότητας με τετραγωνική κορύφωση κοντά στους 72F, μοντελοποιώντας την κίνηση πελατών που οδηγείται από την άνεση.
- Η **Δαπάνη Προώθησης** παρέχει μια κοίλη ανάταση τετραγωνικής ρίζας (φθίνουσες αποδόσεις).
- Μια διακριτή σημαία **Προώθησης** προσθέτει μια παραμετρική επίδραση βήματος στις προωθημένες εβδομάδες.

Οι εβδομαδιαίες `Units` αντλούνται από μια κατανομή Poisson, ταιριάζοντας με τη φύση καταμέτρησης των πωλήσεων μονάδων.

In [1]:
ΔΕΔΟΜΕΝΑ store_sales;
   CALL streaminit(20250531);
   LENGTH Promotion $3;
   ΕΠΑΝΑΛΗΨΗ Week = 1 ΕΩΣ 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      ΕΑΝ rand("uniform") < 0.28 ΤΟΤΕ ΕΠΑΝΑΛΗΨΗ;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      ΤΕΛΟΣ;
      ΑΛΛΙΩΣ ΕΠΑΝΑΛΗΨΗ;
         Promotion  = "No";
         PromoSpend = 0;
      ΤΕΛΟΣ;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      ΕΞΟΔΟΣ;
   ΤΕΛΟΣ;
ΕΚΤΕΛΕΣΗ;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Βήμα 2 — Σκιαγράφηση των προσομοιωμένων δεδομένων

Ένα γρήγορο `PROC MEANS` επιβεβαιώνει ότι οι μεταβλητές καλύπτουν λογικά εύρη λιανικής πριν από τη μοντελοποίηση: τα πλήθη μονάδων είναι μη αρνητικοί ακέραιοι, η τιμή συγκεντρώνεται γύρω από λίγα δολάρια, η θερμοκρασία διατρέχει έναν πλήρη εποχιακό κύκλο, και η δαπάνη προώθησης είναι μηδέν στις μη προωθημένες εβδομάδες.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=store_sales n mean MIN MAX maxdec=2;
   ΜΕΤΑΒΛΗΤΗ Units Price Temperature PromoSpend;
   ΕΤΙΚΕΤΑ Units="Μονάδες" Price="Τιμή" Temperature="Θερμοκρασία" PromoSpend="Δαπάνη Προώθησης";
ΕΚΤΕΛΕΣΗ;

                                                  The MEANS Procedure

 Variable     Label                                   N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------------------------
 Units        Μονάδες                               100           7.03        0.00      103.00
 Price        Τιμή                                  100           3.15        2.81        3.48
 Temperature  Θερμοκρασία                           100          55.50       22.72       83.49
 PromoSpend   Δαπάνη Προώθησης                      100         128.76        0.00      779.00
 ---------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Βήμα 3 — Προσαρμογή του πλήρους προσθετικού μοντέλου ζήτησης

Το βασικό μοντέλο συνδυάζει:

- `param(Promotion)` — μια παραμετρική (γραμμική) επίδραση για τον δείκτη εβδομάδας προώθησης, δηλωμένη στη δήλωση `CLASS`.
- `spline(Price, df=4)` — ένα κυβικό spline εξομάλυνσης που συλλαμβάνει την καμπυλωτή απόκριση τιμής.
- `spline(Temperature, df=4)` — μια ομαλή καμπύλη εποχικότητας.
- `spline(PromoSpend, df=3)` — φθίνουσα ανάταση προώθησης.

Επειδή οι πωλήσεις μονάδων είναι καταμετρήσεις, ορίζουμε `dist=poisson` (σύνδεσμος log). Το `method=gcv` επιτρέπει στη γενικευμένη διασταυρούμενη επικύρωση να καθοδηγήσει την ομαλότητα κάθε συστατικού χωρίς ρητή παράκαμψη βαθμών ελευθερίας. Η δήλωση `OUTPUT` γράφει τις προβλέψεις και τα υπόλοιπα ανά παρατήρηση στο `gam_fit`.

Η διαδικασία αναφέρει μια **Απόκλιση 43.59** έναντι μιας **Μηδενικής Απόκλισης 2041.12** — οι τέσσερις ομαλοί-συν-παραμετρικοί παράγοντες εξηγούν σχεδόν όλη τη μεταβλητότητα που ένα μόνο-σταθερά μοντέλο αφήνει απαρατήρητη — και ένα **AIC 254.61**. Η παραμετρική εκτίμηση `PROMOTIONYES` είναι θετική (+0.41 στην κλίμακα log), επιβεβαιώνοντας την προωθητική ανάταση όγκου, και το spline τιμής φέρει μια έντονα αρνητική γραμμική τάση (−1.71), το χαρακτηριστικό της φθίνουσας ζήτησης.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ gam ΔΕΔΟΜΕΝΑ=store_sales PLOTS=components(CLM commonaxes);
   ΚΛΑΣΗ Promotion;
   ΜΟΝΤΕΛΟ Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   ΕΞΟΔΟΣ out=gam_fit predicted residual;
   ΕΤΙΚΕΤΑ Units="Μονάδες" Promotion="Προώθηση" Price="Τιμή" Temperature="Θερμοκρασία" PromoSpend="Δαπάνη Προώθησης";
ΕΚΤΕΛΕΣΗ;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Μονάδες
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Τιμή)                     4.0000         


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Βήμα 4 — Ένα εστιασμένο μοντέλο τιμής-και-εποχικότητας

Για μια πιο ισχνή ανασκόπηση τιμολόγησης, επαναπροσαρμόζουμε με μόνο τους δύο πιο αποφασιστικής σημασίας ομαλούς παράγοντες — τιμή και θερμοκρασία — δίνοντας στην τιμή επιπλέον ευελιξία (`df=5`) για να επιλύσει τυχόν καμπή κοντά σε ένα ψυχολογικό σημείο τιμής. Η σημαία προώθησης διατηρείται ως παραμετρικός έλεγχος.

Η αφαίρεση της δαπάνης προώθησης αυξάνει την **Απόκλιση σε 62.80** και το **AIC σε 269.82**, και τα δύο υψηλότερα από την Απόκλιση 43.59 και το AIC 254.61 του πλήρους μοντέλου. Ο παραμετρικός όρος `PROMOTIONYES` απορροφά επίσης περισσότερο από το προωθητικό σήμα εδώ (+1.73 έναντι +0.41), αφού το spline δαπάνης δεν είναι πλέον παρόν για να το φέρει. Το spline τιμής διατηρεί την αρνητική του τάση (−1.51), οπότε η βασική ιστορία ζήτησης παραμένει σταθερή σε όλες τις προδιαγραφές.

In [4]:
ΔΙΑΔΙΚΑΣΙΑ gam ΔΕΔΟΜΕΝΑ=store_sales PLOTS=components(CLM);
   ΚΛΑΣΗ Promotion;
   ΜΟΝΤΕΛΟ Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ΕΤΙΚΕΤΑ Units="Μονάδες" Promotion="Προώθηση" Price="Τιμή" Temperature="Θερμοκρασία";
ΕΚΤΕΛΕΣΗ;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Μονάδες
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Τιμή)                     5.0000         5.0000
Spline(Θερμοκρασία)              4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Ερμηνεία των αποτελεσμάτων

Ο πίνακας **Ανάλυση Μοντέλου Παλινδρόμησης** αναφέρει τη γραμμική τάση εντός κάθε συστατικού συν την παραμετρική επίδραση προώθησης. Ο θετικός συντελεστής `PROMOTIONYES` (+0.41 στο πλήρες μοντέλο, +1.73 στο ισχνότερο) επιβεβαιώνει την αναμενόμενη προωθητική ανάταση όγκου, ενώ η αρνητική γραμμική τάση στο spline τιμής (−1.71 και −1.51) αντικατοπτρίζει την κλασική φθίνουσα ζήτηση. Ο μικρός θετικός γραμμικός όρος του spline θερμοκρασίας (+0.03) είναι αναμενόμενος: η πραγματική του ιστορία είναι η καμπυλότητα γύρω από την κορύφωση άνεσης στους 72F, την οποία ένας μόνο γραμμικός συντελεστής δεν μπορεί να συνοψίσει.

Ο πίνακας **Ανάλυση Μοντέλου Εξομάλυνσης** αναφέρει τους βαθμούς ελευθερίας που καταναλώνει κάθε spline. Η τιμή και η θερμοκρασία αντλούν η καθεμία 4 ενεργούς DF (5 για την τιμή στο ισχνότερο μοντέλο) και η δαπάνη προώθησης 3 — καθένα πολύ πάνω από τον έναν DF που θα χρησιμοποιούσε μια ευθεία γραμμή, γι' αυτό ακριβώς μια γραμμική παλινδρόμηση θα παρέβλεπε αυτές τις καμπυλωτές σχέσεις.

Οι **Στατιστικές Προσαρμογής** επιτρέπουν τη σύγκριση των δύο προδιαγραφών απευθείας. Το πλήρες μοντέλο τεσσάρων παραγόντων εμφανίζει Απόκλιση 43.59 και AIC 254.61 έναντι 62.80 και 269.82 του ισχνότερου μοντέλου· και τα δύο κριτήρια ευνοούν το πλήρες μοντέλο, δείχνοντας ότι η δαπάνη προώθησης και η σημαία προώθησης προσθέτουν επεξηγηματική ισχύ πέρα από την τιμή και τη θερμοκρασία μόνες τους. Σε σχέση με τη Μηδενική Απόκλιση 2041.12, και τα δύο μοντέλα συλλαμβάνουν τη συντριπτική πλειοψηφία της μεταβλητότητας ζήτησης.

Μαζί, αυτοί οι πίνακες δίνουν σε έναν διευθυντή κατηγορίας μια ποσοτικοποιημένη, βασισμένη σε δεδομένα ιστορία ζήτησης: μια απότομη απόκριση τιμής που ενημερώνει το βάθος έκπτωσης, ένα εποχιακό παράθυρο θερμοκρασίας, και μια επίδραση δαπάνης προώθησης φθίνουσας απόδοσης — πολύ πιο ακριβή καθοδήγηση από μια απλή γραμμική εκτίμηση ελαστικότητας. (Το PROC GAM δέχεται επίσης `plots=components` για να αποδώσει τις καμπύλες μερικής πρόβλεψης για κάθε ομαλό όρο· οι αριθμητικοί πίνακες συστατικών παραπάνω είναι η πηγή από την οποία σχεδιάζονται αυτές οι καμπύλες.)